# Xarray-spatial
### User Guide: Flood analysis
-----
The hydrology tools in the [previous guide](11_Hydrology.ipynb) answer "where does water go?" This guide picks up where that left off and asks "what actually floods?"

We'll walk through a naive flood analysis using four functions from `xrspatial.flood`:

1. Compute HAND (Height Above Nearest Drainage) from the hydrology stack
2. [Inundation mapping](#Inundation-mapping): mark which cells flood at a given water level
3. [Flood depth](#Flood-depth): compute water depth at each flooded cell
4. [Curve number runoff](#Curve-number-runoff): estimate how much rainfall becomes runoff using the SCS/NRCS method
5. [Travel time](#Travel-time): estimate how long it takes water to reach the outlet

"Naive" because we're using a single static water level across the whole grid, uniform curve numbers, and simplified Manning's velocity. Real flood modeling needs event hydrographs, channel geometry, and spatially varying land cover. But this gets you surprisingly far for screening-level analysis.

-----------

In [ ]:
import numpy as np
import xarray as xr

from datashader.transfer_functions import shade, stack, set_background
from datashader.colors import Elevation

import xrspatial
from xrspatial.flood import flood_depth, inundation, curve_number_runoff, travel_time

## Prepare the terrain

Same setup as the hydrology guide: load the Copernicus 30m DEM tile, fill depressions, compute flow direction and accumulation. If the remote tile isn't available, we fall back to synthetic terrain.

In [ ]:
try:
    import rasterio
    from rasterio.windows import Window

    url = (
        "https://copernicus-dem-30m.s3.amazonaws.com/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM/"
        "Copernicus_DSM_COG_10_N46_00_W123_00_DEM.tif"
    )

    with rasterio.open(url) as src:
        window = Window(col_off=2400, row_off=2400, width=600, height=600)
        data = src.read(1, window=window).astype(np.float64)
        nodata = src.nodata

    if nodata is not None:
        data[data == nodata] = np.nan

    H, W = data.shape
    dem = xr.DataArray(data, dims=['y', 'x'], name='elevation',
                       attrs={'res': (1, 1)})
    dem['y'] = np.linspace(H - 1, 0, H)
    dem['x'] = np.linspace(0, W - 1, W)
    print(f"Loaded Copernicus 30m DEM: {dem.shape}")

except Exception as e:
    print(f"Remote DEM unavailable ({e}), generating synthetic terrain")
    H, W = 600, 600
    dem = xr.DataArray(np.zeros((H, W)), dims=['y', 'x'])
    dem = dem.xrs.generate_terrain(seed=10)
    dem.name = 'elevation'
    print(f"Generated terrain: {dem.shape}")

In [ ]:
# Fill depressions, resolve flats, compute flow direction and accumulation.
# This is the same preprocessing from the hydrology guide.
dem_filled = xrspatial.fill(dem)

rng = np.random.RandomState(42)
dem_filled.values += rng.uniform(0, 0.001, dem_filled.shape)
dem_filled = xrspatial.fill(dem_filled)
rng2 = np.random.RandomState(123)
dem_filled.values += rng2.uniform(0, 1e-6, dem_filled.shape)

flow_dir = xrspatial.flow_direction(dem_filled)
flow_accum = xrspatial.flow_accumulation(flow_dir)

print(f"Flow accumulation range: {np.nanmin(flow_accum.values):.0f} to "
      f"{np.nanmax(flow_accum.values):.0f}")

In [ ]:
# Build a basemap we'll reuse throughout
hillshade = dem.xrs.hillshade()
hillshade_img = shade(hillshade, cmap=['gray', 'white'], how='linear')
terrain_img = shade(dem, cmap=Elevation, alpha=128, how='linear')

# Also render the stream network for context
threshold = 200
streams = flow_accum.copy()
streams.values = np.where(streams.values >= threshold, streams.values, np.nan)

stack(hillshade_img, terrain_img,
      shade(streams, cmap=['lightblue', 'darkblue'], alpha=220, how='log'))

## Compute HAND

HAND (Height Above Nearest Drainage) measures the vertical distance from each cell to the stream channel it drains to. Cells on the channel have HAND near zero; cells on ridges have high values.

This is what the flood functions use as input: a cell floods when the water level in the channel exceeds its HAND value.

In [ ]:
hand_raster = xrspatial.hand(flow_dir, flow_accum, dem_filled, threshold=threshold)

print(f"HAND range: {np.nanmin(hand_raster.values):.1f} to "
      f"{np.nanmax(hand_raster.values):.1f} m")

stack(hillshade_img,
      shade(hand_raster, cmap=['darkblue', 'cyan', 'yellow', 'red'],
            alpha=160, how='linear'))

## Inundation mapping

The simplest question: which cells flood?

`inundation(hand_raster, water_level)` returns 1.0 where HAND <= water_level, 0.0 elsewhere. NaN cells stay NaN.

We'll try a few water levels to see how the flood extent grows. In the real world, a 5m water level means the channel is running 5 metres above its normal level, which would be a major flood event for most streams.

In [ ]:
for wl in [2, 5, 10, 20]:
    mask = inundation(hand_raster, water_level=wl)
    n_flooded = int(np.nansum(mask.values))
    pct = 100 * n_flooded / (H * W)
    print(f"Water level {wl:2d} m: {n_flooded:>6d} cells flooded ({pct:.1f}%)")

In [ ]:
# Visualize the 10m flood extent
flood_mask = inundation(hand_raster, water_level=10)

# Replace 0s with NaN so only flooded cells render
flood_viz = flood_mask.copy()
flood_viz.values = np.where(flood_viz.values == 1.0, 1.0, np.nan)

stack(hillshade_img, terrain_img,
      shade(flood_viz, cmap='blue', alpha=160, how='linear'))

## Flood depth

Knowing *which* cells flood is useful, but for damage estimation you need to know *how deep*.

`flood_depth(hand_raster, water_level)` returns `water_level - HAND` where the cell is inundated, NaN elsewhere. A cell with HAND=3m under a 10m water level has 7m of water above it.

In [ ]:
depth = flood_depth(hand_raster, water_level=10)

# Stats on flooded cells only
flooded_vals = depth.values[~np.isnan(depth.values)]
print(f"Flood depth at water_level=10m:")
print(f"  min:    {flooded_vals.min():.2f} m")
print(f"  median: {np.median(flooded_vals):.2f} m")
print(f"  max:    {flooded_vals.max():.2f} m")

stack(hillshade_img, terrain_img,
      shade(depth, cmap=['lightyellow', 'orange', 'darkred'],
            alpha=180, how='linear'))

## Curve number runoff

How much of a rainstorm actually becomes runoff? The SCS/NRCS curve number method gives a quick estimate:

```
S  = (25400 / CN) - 254        # potential maximum retention (mm)
Ia = 0.2 * S                    # initial abstraction (mm)
Q  = (P - Ia)^2 / (P + 0.8*S)  # runoff (mm), only where P > Ia
```

CN ranges from 0 to 100. Low CN (30-50) means permeable soil and good ground cover, so most rain soaks in. High CN (80-98) means impervious surfaces or saturated soil, so most rain runs off.

Here we'll apply a uniform rainfall of 100mm (a heavy storm) across the grid and compare runoff under different land cover assumptions.

In [ ]:
# 100mm rainfall everywhere
rainfall = xr.DataArray(
    np.full((H, W), 100.0, dtype=np.float64),
    dims=['y', 'x'], coords=dem.coords,
    attrs=dem.attrs, name='rainfall',
)

# Compare CN values: forest (55), pasture (70), suburban (80), pavement (95)
for label, cn in [('forest', 55), ('pasture', 70), ('suburban', 80), ('pavement', 95)]:
    q = curve_number_runoff(rainfall, curve_number=cn)
    print(f"CN={cn:2d} ({label:>9s}): {q.values[0, 0]:.1f} mm runoff from 100 mm rain")

The function also accepts a spatially varying CN raster. In practice you'd derive this from a land cover map. Here we'll fake one: valley floors (low HAND) get a higher CN (more impervious, developed land) and hillsides get a lower CN (forested).

In [ ]:
# Simple CN map: low HAND -> urban-ish (CN=85), high HAND -> forested (CN=55)
hand_vals = hand_raster.values.copy()
hand_vals[np.isnan(hand_vals)] = 0
cn_data = np.where(hand_vals < 20, 85.0, 55.0)
cn_raster = xr.DataArray(cn_data, dims=['y', 'x'], coords=dem.coords,
                         attrs=dem.attrs, name='curve_number')

runoff = curve_number_runoff(rainfall, curve_number=cn_raster)

print(f"Spatially varying runoff:")
print(f"  valley (CN=85): {runoff.values[cn_data == 85].mean():.1f} mm")
print(f"  hills  (CN=55): {runoff.values[cn_data == 55].mean():.1f} mm")

stack(hillshade_img,
      shade(runoff, cmap=['lightyellow', 'orange', 'darkred'],
            alpha=180, how='linear'))

## Travel time

How long does it take runoff to reach the outlet? `travel_time` estimates this with a simplified Manning's equation:

```
velocity    = (1/n) * sqrt(tan(slope))
travel_time = flow_length / velocity
```

where `n` is Manning's roughness coefficient. The maximum travel time across the grid is the *time of concentration*, which tells you roughly how long after a rainstorm the peak flow arrives at the outlet.

We need slope and flow length as inputs. Near-zero slopes get clamped to `tan(0.001 deg)` to avoid division by zero (same approach as TWI).

In [ ]:
slope_raster = xrspatial.slope(dem_filled)
fl_raster = xrspatial.flow_length(flow_dir)

print(f"Slope range: {np.nanmin(slope_raster.values):.2f} to "
      f"{np.nanmax(slope_raster.values):.1f} degrees")
print(f"Flow length range: {np.nanmin(fl_raster.values):.0f} to "
      f"{np.nanmax(fl_raster.values):.0f} cells")

In [ ]:
# Manning's n = 0.03 is a reasonable value for natural channels.
# Try 0.06 for heavily vegetated overland flow.
tt = travel_time(fl_raster, slope_raster, mannings_n=0.03)

tt_vals = tt.values[~np.isnan(tt.values)]
tc = tt_vals.max()
print(f"Travel time range: {tt_vals.min():.1f} to {tc:.1f}")
print(f"Time of concentration (max travel time): {tc:.1f}")

stack(hillshade_img,
      shade(tt, cmap=['darkblue', 'cyan', 'yellow', 'red'],
            alpha=160, how='eq_hist'))

In [ ]:
# Compare roughness: smoother surface (channel) vs rougher (vegetated hillside)
tt_smooth = travel_time(fl_raster, slope_raster, mannings_n=0.03)
tt_rough  = travel_time(fl_raster, slope_raster, mannings_n=0.06)

print(f"Time of concentration:")
print(f"  n=0.03 (channel):    {np.nanmax(tt_smooth.values):.1f}")
print(f"  n=0.06 (vegetated):  {np.nanmax(tt_rough.values):.1f}")
print(f"  ratio: {np.nanmax(tt_rough.values) / np.nanmax(tt_smooth.values):.2f}x")

## Putting it together

In practice you'd run these in sequence: estimate how much rain becomes runoff (`curve_number_runoff`), pick a water level scenario, map the flood extent and depth (`inundation`, `flood_depth`), and check how quickly water arrives at the outlet (`travel_time`).

Below we overlay flood depth on the basemap, coloured from yellow (shallow) to dark red (deep), with the stream network in blue.

In [ ]:
# Final composite: 10m flood scenario
depth_10m = flood_depth(hand_raster, water_level=10)

n_flooded = int(np.sum(~np.isnan(depth_10m.values)))
pct_flooded = 100 * n_flooded / (H * W)
print(f"10m flood scenario: {n_flooded} cells flooded ({pct_flooded:.1f}% of grid)")
print(f"Median depth in flooded area: {np.nanmedian(depth_10m.values):.1f} m")

stack(
    hillshade_img,
    terrain_img,
    shade(depth_10m, cmap=['lightyellow', 'orange', 'darkred'],
          alpha=200, how='linear'),
    shade(streams, cmap='blue', alpha=100, how='log'),
)

## Caveats

What we did here is useful for screening but has real limitations.

We applied the same water level to every channel reach. In reality, water levels vary along the network depending on contributing area, channel slope, and upstream conditions. We also didn't simulate how a flood wave propagates downstream over time; that requires a hydrograph router (e.g. Muskingum-Cunge), which is outside the scope of these tools.

Manning's n varies with land cover, so a spatially varying DataArray for `mannings_n` would give better travel time estimates than the uniform scalar we used. And 30m DEM cells can miss narrow floodplains and underestimate flood extent in tight valleys. Higher-resolution LiDAR DEMs (1-5m) work better for local studies.

The SCS curve number method itself was designed for small agricultural watersheds. It assumes a single-peak storm and doesn't account for snowmelt, baseflow, or soil moisture dynamics.

For anything beyond screening, pair these tools with a proper hydraulic model (HEC-RAS, LISFLOOD, etc.).

## References

- Nobre, A.D. et al. (2011). HAND, a new terrain descriptor using SRTM-DEM. *Mapping accuracy assessment for the Amazon basin*. Remote Sensing of Environment, 112(9), 3469-3481.
- USDA-NRCS (1986). Urban Hydrology for Small Watersheds, TR-55.
- Manning, R. (1891). On the flow of water in open channels and pipes. *Transactions of the Institution of Civil Engineers of Ireland*, 20, 161-207.
- Copernicus DEM on AWS: https://registry.opendata.aws/copernicus-dem/